# Caching (Enterprise AI System Design)

Caching is one of the **highest ROI optimizations** in Enterprise AI because it reduces:

- LLM Cost
- Response Time
- Database Load
- Vector Search Load

Interviewers ask:

- What should you cache?
- Where should you cache?
- Redis vs Cache?
- Cache Aside Pattern?
- What is Cache Invalidation?
- When should you NOT cache?

---

# 1. What is Caching?

## Definition

Caching is the process of storing **frequently accessed data in a fast storage layer** (usually Redis) so that future requests can be served quickly without recomputing or fetching the data again.

---

## Interview Answer

> "Caching stores frequently accessed data in a high-speed storage layer such as Redis to reduce latency, improve throughput, minimize database and LLM calls, and lower infrastructure costs."

---

# Azure vs AWS

| Azure | AWS |
|--------|-----|
| Azure Cache for Redis | Amazon ElastiCache (Redis) |

---

# 2. Why Caching?

Suppose

1000 users ask

```text id="4i1k4n"
What is the leave policy?
```

Without Cache

```text id="nnr51o"
User

↓

FastAPI

↓

LangGraph

↓

Retriever

↓

Qdrant

↓

Bedrock

↓

Response
```

1000 Bedrock Calls

Expensive

---

With Cache

```text id="yjlwmx"
User

↓

Redis

↓

Found

↓

Return

(No LLM)
```

---

# 3. Enterprise Architecture

```text id="ccg3jo"
                           User
                             │
                             ▼
 Azure Front Door / Route53 + CloudFront
                             │
                             ▼
Azure API Management / Amazon API Gateway
                             │
                             ▼
Azure App Gateway / AWS ALB
                             │
                             ▼
FastAPI (Container Apps) / ECS Fargate
                             │
                  ┌──────────┴──────────┐
                  ▼                     ▼
        Redis / ElastiCache       LangGraph
                  │                     │
           Cache Hit?                  ▼
                  │             Retriever
          Yes ────┘                  │
                  │                  ▼
                  ▼          Qdrant / OpenSearch
           Return Answer             │
                                     ▼
                          Azure OpenAI / Bedrock
                                     │
                                     ▼
                            Save to Cache
```

---

# 4. Cache Flow

```text id="0a8thn"
User

↓

Redis

↓

Hit?

↓

Yes

↓

Return

↓

No

↓

LLM

↓

Save

↓

Redis

↓

Return
```

---

# 5. What Should We Cache?

## LLM Responses ⭐⭐⭐⭐⭐

Question

↓

Answer

---

## Embeddings ⭐⭐⭐⭐

Same document

↓

Embedding

↓

Redis

---

## User Session ⭐⭐⭐⭐

Login

↓

Redis

---

## Frequently Used Metadata ⭐⭐⭐

Department List

Role List

Configuration

---

## Prompt Templates ⭐⭐⭐

Instead of reading every time

↓

Cache

---

# 6. What Should NOT Be Cached?

❌ Passwords

❌ JWT Secrets

❌ Frequently changing financial transactions

❌ Sensitive personal information (unless encrypted and justified)

❌ Large PDFs

---

# 7. Cache Patterns

## Cache Aside (Most Common)

Application checks cache first.

```text id="jvqsn4"
Application

↓

Redis

↓

Found?

↓

Yes

↓

Return

↓

No

↓

Database / Bedrock

↓

Redis

↓

Return
```

This is the most common pattern.

---

## Write Through

```text id="8pcrq6"
Application

↓

Redis

↓

Database
```

Both updated together.

---

## Write Back

```text id="zhtwnr"
Application

↓

Redis

↓

Later

↓

Database
```

Very fast.

---

# 8. Coding Example

Install

```bash id="eywdvr"
pip install redis
```

---

Simple Example

```python id="1xqh4t"
import redis

cache = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

cache.set("company", "EPAM")

print(cache.get("company"))
```

Output

```text id="5nl3s3"
EPAM
```

---

# 9. LLM Cache Example

```python id="k97k2z"
question = "What is RAG?"

answer = cache.get(question)

if answer:
    print("Cache Hit")
else:
    answer = bedrock.generate(question)
    cache.set(question, answer, ex=3600)
```

---

# 10. Cache TTL

TTL

↓

Time To Live

Example

```python id="w2j4d2"
cache.set(
    "policy",
    answer,
    ex=3600
)
```

Expires

after

1 hour.

---

# 11. Cache Invalidation

One of the most asked questions.

Suppose

Leave Policy

changes.

Old Cache

↓

Wrong Answer.

Solution

Delete Cache.

```python id="hbfwfx"
cache.delete("leave_policy")
```

---

# 12. Cache Hit vs Miss

Example

100 Requests

80

↓

Cache Hit

20

↓

LLM

Hit Rate

80%

Higher hit rates generally reduce cost and latency.

---

# 13. Redis vs Cache

Interview Trick

Redis

↓

Technology

Cache

↓

Concept

Redis

implements

Cache.

---

# 14. Multi-Level Cache

```text id="m4lzaz"
Browser Cache

↓

CloudFront

↓

Redis

↓

Database
```

Each level

reduces

latency.

---

# 15. Best Practices

✅ Cache LLM Responses

✅ Use TTL

✅ Cache Sessions

✅ Monitor Cache Hit Ratio

✅ Invalidate on Updates

---

# 16. Common Mistakes

❌ No TTL

❌ Never Invalidate

❌ Cache Everything

❌ Cache Large Files

❌ Cache Highly Dynamic Data

---

# 17. Interview Questions

### Q1. Why Cache?

Reduce latency and cost.

---

### Q2. What should you cache?

- LLM Responses
- Sessions
- Metadata
- Embeddings (if recomputed frequently)

---

### Q3. What should not be cached?

Large files

Passwords

Highly dynamic data

---

### Q4. What is TTL?

Automatic expiration.

---

### Q5. What is Cache Hit?

Data found in cache.

---

### Q6. What is Cache Miss?

Data not found.

Need

Database

or

LLM.

---

### Q7. What is Cache Invalidation?

Removing or refreshing stale cached data after the source data changes.

---

# 18. Scenario-Based Question

### Interviewer

> HR updates the leave policy document. Users still receive the old answer. Why?

Expected Answer

The cached response is stale.

Invalidate the Redis entry (or use versioned cache keys), regenerate embeddings if the source document changed, update the vector database, and repopulate the cache with fresh results.

---

# 19. Complete Enterprise Flow

```text id="sjlwmc"
User
 │
 ▼
FastAPI
 │
 ▼
Redis / ElastiCache
 │
 ├── Hit
 │      │
 │      ▼
 │   Return
 │
 └── Miss
        │
        ▼
LangGraph
        │
        ▼
Retriever
        │
        ▼
Qdrant / OpenSearch
        │
        ▼
Bedrock / Azure OpenAI
        │
        ▼
Save Cache (TTL)
        │
        ▼
Return
```

---

# 20. Cache vs Redis vs PostgreSQL vs Qdrant

| Component | Purpose |
|-----------|---------|
| Redis | Cache, Sessions |
| PostgreSQL | Permanent Business Data |
| Qdrant / OpenSearch | Embeddings |
| Amazon S3 | Files |

---

# 21. EPAM Senior Answer (2–3 Minutes)

> "In enterprise AI applications, I use Redis through Amazon ElastiCache or Azure Cache for Redis as the primary caching layer. The application follows the Cache-Aside pattern: before invoking LangGraph and the LLM, it checks Redis for a cached response. If a cache hit occurs, the response is returned immediately. On a cache miss, the workflow retrieves relevant documents, invokes AWS Bedrock or Azure OpenAI, returns the generated answer, and stores it in Redis with an appropriate TTL. I also use Redis for session management, frequently accessed metadata, and selected embedding caches when beneficial. To maintain consistency, I invalidate or refresh cached entries whenever underlying documents change. I monitor cache hit ratio, latency, and memory usage to ensure the cache remains effective and cost-efficient in production."